# Feature Table Inspection

モデル構築に使用する施設のデータと、モデルの評価（external evaluation）に使う施設のデータを分割して保存する

## Set Config

In [1]:
# ライブラリのインポート
import os
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import seaborn as sns

In [2]:
# フォルダ設定
data_dir = '../data/'
output_dir = '../output/'

# date
date = '20260202_emergency'

# 特徴量tableのファイル名
feature_table = f'{date}_features.csv'

In [3]:
# 保存するファイル名の指定
df_development_file_name = f'{date}_df_development_raw.csv'
# df_evaluation_file_name = f'{date}_df_evaluation_raw.csv'

## Load data

In [4]:
df = pl.read_csv(os.path.join(data_dir + feature_table))

In [5]:
# record数と症例数を確認
print('Number of records:', len(df))
print('Number of icu_stay_id:', len(df['icu_stay_id'].unique()))
print('Average number of records per icu_stay_id:', len(df) / len(df['icu_stay_id'].unique()))

Number of records: 4899756
Number of icu_stay_id: 41300
Average number of records per icu_stay_id: 118.6381598062954


In [6]:

# SEED = 42  # 再現性のため
# 1) ICU滞在IDのユニーク集合を作る
# ids = df.select("icu_stay_id").unique()
# n_ids = ids.height

# 2) 1/100だけサンプル（最低1件は確保）
# n_sample = max(1, n_ids // 50)
# ids_sampled = ids.sample(n=n_sample, shuffle=True, seed=SEED)

# 3) 選ばれたIDの症例のみ残す（semi join）
# df_sampled = df.join(ids_sampled, on="icu_stay_id", how="semi")

# record数と症例数を確認
# print('Number of records:', len(df_sampled))
# print('Number of icu_stay_id:', len(df_sampled['icu_stay_id'].unique()))
# print('Average number of records per icu_stay_id:', len(df_sampled) / len(df_sampled['icu_stay_id'].unique()))

# Inspect Feature Table

In [7]:
def describe_df(df: pl.DataFrame, start: int, end: int) -> pl.DataFrame:
    return df[:, start:end].describe()

In [8]:
df.head()

icu_stay_id,time_window_index,start_time,end_time,delirium_flag,label_delirium_indication,hospital_id,icu_admit_time,female,age,icu_admission_type,diagnosis_category,covid19,sepsis,stroke,poisoning,anaphylaxis,burn,temperature_disorder,hanging_asphyxiation,trauma,infection,neoplasms,circulatory,respiratory,digestive,neurological,genitourinary,electrolyte_metabolic_disorders,musculoskeletal,psychiatric,congenital,undefined,other,icd10,apache2_score,charlson_comorbidity_index,…,aspartate_aminotransferase,blood_urea_nitrogen,alanine_aminotransferase,total_bilirubin,crp,albumin,rass_score,respiration_24hours,coagulation_24hours,liver_24hours,cardiovascular_24hours,cns_24hours,renal_24hours,sofa_24hours,input_amount,cumsum_in,window_total_output,cumsum_out,cumsum_balance,active_ingredient_propofol,active_ingredient_noradrenaline,active_ingredient_fentanyl,active_ingredient_dexmedetomidine,active_ingredient_dopamine,active_ingredient_midazolam,active_ingredient_dobutamine,active_ingredient_landiolol,active_ingredient_vasopressin,active_ingredient_remifentanil,active_ingredient_thiamylal,active_ingredient_adrenaline,active_ingredient_vecuronium,active_ingredient_rocuronium,transfusion_rbc,transfusion_platelet,transfusion_plasma,transfusion_albumin
i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,…,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1021325852177154048,21,"""2200-01-02 09:00:00 UTC""","""2200-01-02 10:00:00 UTC""",0,0,1,12,1,83,"""emergency""","""circulatory""",0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,"""I210""",24,5,…,56.0,20.0,24.0,0.8,0.612,3.0,-1.0,1,1,0,3,1,4,10,131.0,3459.6,115.0,1154.996,2304.604,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1021325852177154048,22,"""2200-01-02 10:00:00 UTC""","""2200-01-02 11:00:00 UTC""",0,0,1,12,1,83,"""emergency""","""circulatory""",0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,"""I210""",24,5,…,56.0,20.0,24.0,0.8,0.612,3.0,-1.0,1,1,0,3,1,4,10,131.0,3590.6,115.0,1269.996,2320.604,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1021325852177154048,23,"""2200-01-02 11:00:00 UTC""","""2200-01-02 12:00:00 UTC""",0,0,1,12,1,83,"""emergency""","""circulatory""",0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,"""I210""",24,5,…,56.0,20.0,24.0,0.8,0.612,3.0,-1.0,1,1,0,3,1,4,10,131.0,3721.6,115.0,1384.996,2336.604,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1021325852177154048,24,"""2200-01-02 12:00:00 UTC""","""2200-01-02 13:00:00 UTC""",0,0,1,12,1,83,"""emergency""","""circulatory""",0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,"""I210""",24,5,…,56.0,20.0,24.0,0.8,0.612,3.0,-1.0,1,1,0,3,1,4,10,131.0,3852.6,115.0,1499.996,2352.604,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1021325852177154048,25,"""2200-01-02 13:00:00 UTC""","""2200-01-02 14:00:00 UTC""",0,0,1,12,1,83,"""emergency""","""circulatory""",0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,"""I210""",24,5,…,56.0,20.0,24.0,0.8,0.612,3.0,-1.0,1,1,0,3,1,4,10,131.0,3983.6,140.0,1639.996,2343.604,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
df["label_delirium_indication"].describe()

statistic,value
str,f64
"""count""",4.899756e6
"""null_count""",0.0
"""mean""",0.190427
"""std""",0.392638
"""min""",0.0
"""25%""",0.0
"""50%""",0.0
"""75%""",0.0
"""max""",1.0


In [10]:
describe_df(df, 0, 10)

statistic,icu_stay_id,time_window_index,start_time,end_time,delirium_flag,label_delirium_indication,hospital_id,icu_admit_time,female,age
str,f64,f64,str,str,f64,f64,f64,f64,f64,f64
"""count""",4.899756e6,4.899756e6,"""4899756""","""4899756""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",5.7932e17,227.440157,null,null,0.002515,0.190427,4.449725,13.524164,0.367817,65.512759
"""std""",3.3216e17,387.256886,null,null,0.050083,0.392638,3.526787,5.827828,0.482211,17.288585
"""min""",2.8023e13,0.0,"""2200-01-01 00:00:00 UTC""","""2200-01-01 01:00:00 UTC""",0.0,0.0,1.0,0.0,0.0,15.0
"""25%""",2.9493e17,38.0,null,null,0.0,0.0,1.0,10.0,0.0,55.0
"""50%""",5.8050e17,109.0,null,null,0.0,0.0,3.0,14.0,0.0,70.0
"""75%""",8.6824e17,264.0,null,null,0.0,0.0,7.0,18.0,1.0,78.0
"""max""",1.1528e18,8209.0,"""2211-11-06 09:00:00 UTC""","""2211-11-06 10:00:00 UTC""",1.0,1.0,11.0,23.0,1.0,99.0


In [11]:
describe_df(df, 10, 20)

statistic,icu_admission_type,diagnosis_category,covid19,sepsis,stroke,poisoning,anaphylaxis,burn,temperature_disorder,hanging_asphyxiation
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""4899756""","""4899756""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,0.091321,0.028948,0.069882,0.011124,0.000665,0.013327,0.011987,0.001881
"""std""",null,null,0.288065,0.16766,0.254947,0.10488,0.025774,0.114672,0.108827,0.043333
"""min""","""emergency""","""anaphylaxis""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""","""emergency""","""trauma""",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [12]:
describe_df(df, 20, 30)

statistic,trauma,infection,neoplasms,circulatory,respiratory,digestive,neurological,genitourinary,electrolyte_metabolic_disorders,musculoskeletal
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.134807,0.107013,0.045766,0.213172,0.049925,0.062186,0.025841,0.018733,0.038843,0.021927
"""std""",0.341517,0.309129,0.208978,0.409549,0.21779,0.241493,0.158661,0.135582,0.193221,0.146446
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [13]:
describe_df(df, 30, 40)

statistic,psychiatric,congenital,undefined,other,icd10,apache2_score,charlson_comorbidity_index,myocardial_infarct,congestive_heart_failure,peripheral_arterial_disease
str,f64,f64,f64,f64,str,f64,f64,f64,f64,f64
"""count""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,"""4832682""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""",0.0,0.0,0.0,0.0,"""67074""",0.0,0.0,0.0,0.0,0.0
"""mean""",0.008715,0.002619,0.018575,0.022743,null,28.833738,3.217867,0.045141,0.107883,0.065867
"""std""",0.092948,0.051106,0.13502,0.149083,null,9.744567,2.728794,0.207614,0.310233,0.24805
"""min""",0.0,0.0,0.0,0.0,"""<TBD>""",1.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,null,22.0,2.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,null,29.0,3.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,null,36.0,4.0,0.0,0.0,0.0
"""max""",1.0,1.0,1.0,1.0,"""Z988""",61.0,25.0,1.0,1.0,1.0


In [14]:
describe_df(df, 40, 50)

statistic,cerebrovascular_disease,dementia,chronic_pulmonary_disease,connective_tissue_disease,peptic_ulcer_disease,mild_liver_disease,diabetes_without_cc,diabetes_with_cc,paraplegia,renal_disease
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.101155,0.009834,0.036346,0.02741,0.100157,0.048867,0.032106,0.019691,0.006175,0.049488
"""std""",0.301534,0.098677,0.187149,0.163275,0.300209,0.215589,0.176281,0.138935,0.078339,0.216884
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [15]:
describe_df(df, 50, 60)

statistic,malignant_cancer,severe_liver_disease,metastatic_solid_tumor,aids,age_score,current_time_of_day,fio2_ordered,fio2_ordered_is_current_value,delta_fio2_ordered_lag1,delta_fio2_ordered_lag6
str,f64,f64,f64,f64,f64,f64,str,f64,str,str
"""count""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,"""3173278""",4.899756e6,"""2156921""","""2042117"""
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,"""1726478""",0.0,"""2742835""","""2857639"""
"""mean""",0.094231,0.010084,0.021831,0.006495,2.15055,11.423817,null,0.445576,null,null
"""std""",0.29215,0.099912,0.146132,0.08033,1.38484,6.949764,null,0.497029,null,null
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,"""100.0""",0.0,"""-1.0""","""-1.0"""
"""25%""",0.0,0.0,0.0,0.0,1.0,5.0,null,0.0,null,null
"""50%""",0.0,0.0,0.0,0.0,2.0,11.0,null,0.0,null,null
"""75%""",0.0,0.0,0.0,0.0,3.0,17.0,null,1.0,null,null
"""max""",1.0,1.0,1.0,1.0,4.0,23.0,"""99.0""",1.0,"""9.999999999999993""","""9.999999999999993"""


In [16]:
describe_df(df, 60, 70)

statistic,delta_fio2_ordered_lag12,peep_ordered,peep_ordered_is_current_value,delta_peep_ordered_lag1,delta_peep_ordered_lag6,delta_peep_ordered_lag12,inspiratory_pressure,time_since_last_control_ventilation,mv_duration,ph
str,str,str,f64,str,str,str,str,f64,f64,f64
"""count""","""1931919""","""3164739""",4.899756e6,"""2156306""","""2040618""","""1929564""","""2835259""",3.565127e6,4.899756e6,3.452868e6
"""null_count""","""2967837""","""1735017""",0.0,"""2743450""","""2859138""","""2970192""","""2064497""",1.334629e6,0.0,1.446888e6
"""mean""",null,null,0.445483,null,null,null,null,51.53172,75.632598,7.422221
"""std""",null,null,0.497019,null,null,null,null,167.507295,213.905955,0.072504
"""min""","""-1.0""","""-2.0""",0.0,"""-0.09872144461049981""","""-0.09872144461049981""","""-0.09872144461049981""","""-1.0""",0.0,0.0,6.343
"""25%""",null,null,0.0,null,null,null,null,0.0,0.0,7.39
"""50%""",null,null,0.0,null,null,null,null,0.0,0.0,7.431
"""75%""",null,null,1.0,null,null,null,null,23.0,73.0,7.467
"""max""","""9.999999999999993""","""NaN""",1.0,"""NaN""","""NaN""","""NaN""","""NaN""",3815.0,6590.0,7.984


In [17]:
describe_df(df, 70, 80)

statistic,pao2,pvo2,paco2,pvco2,bg_lactate,bg_glucose,gcs_e,gcs_v,gcs_m,body_weight
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",3.286853e6,337890.0,3.290348e6,338130.0,3.452591e6,3.115766e6,4.80733e6,4.807264e6,4.807189e6,4.206564e6
"""null_count""",1.612903e6,4.561866e6,1.609408e6,4.561626e6,1.447165e6,1.78399e6,92426.0,92492.0,92567.0,693192.0
"""mean""",103.69034,46.254681,38.840878,43.636056,1.55235,152.543647,2.923816,2.774542,4.633805,60.580891
"""std""",47.083938,23.550839,8.837953,11.597468,1.584281,54.100745,1.233801,1.889652,1.943084,15.928597
"""min""",8.0,2.7,5.0,8.9,0.0,0.0,1.0,1.0,1.0,1.0
"""25%""",77.3,33.2,33.7,37.1,0.83,117.0,2.0,1.0,4.0,49.7
"""50%""",92.6,41.8,37.8,42.1,1.2,141.0,3.0,1.0,6.0,58.9
"""75%""",117.0,53.4,42.4,47.6,1.7,176.0,4.0,5.0,6.0,69.4
"""max""",1592.0,560.0,754.0,197.0,47.0,1617.0,4.0,5.0,6.0,200.0


In [18]:
describe_df(df, 80, 90)

statistic,bt50,hr10,hr50,hr90,hr_sd,delta_hr10_lag1,delta_hr10_lag6,delta_hr10_lag12,delta_hr50_lag1,delta_hr50_lag6
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.531271e6,4.7702e6,4.7702e6,4.7702e6,4.715409e6,4.65112e6,4.441742e6,4.221078e6,4.65112e6,4.441742e6
"""null_count""",368485.0,129556.0,129556.0,129556.0,184347.0,248636.0,458014.0,678678.0,248636.0,458014.0
"""mean""",37.152272,80.502817,84.494434,89.677684,4.176087,-0.084192,-0.332805,-0.503413,-0.09861,-0.341167
"""std""",1.016618,18.557457,19.069825,19.973881,3.581609,6.261587,10.973488,13.020507,6.174778,11.397683
"""min""",10.0,0.0,0.0,0.0,0.0,-190.0,-183.0,-218.0,-205.5,-203.0
"""25%""",36.6,67.0,71.0,76.0,2.019328,-2.0,-5.0,-7.0,-2.0,-5.5
"""50%""",37.1,79.0,83.0,88.0,3.341064,0.0,0.0,0.0,0.0,0.0
"""75%""",37.7,92.0,96.0,102.0,5.276105,2.0,4.9,6.0,2.0,5.0
"""max""",45.0,267.0,269.0,294.0,134.819014,192.0,182.0,184.0,203.0,203.0


In [19]:
describe_df(df, 90, 100)

statistic,delta_hr50_lag12,delta_hr90_lag1,delta_hr90_lag6,delta_hr90_lag12,delta_hr_sd_lag1,delta_hr_sd_lag6,delta_hr_sd_lag12,rr10,rr50,rr90
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.221078e6,4.65112e6,4.441742e6,4.221078e6,4.614804e6,4.403164e6,4.181261e6,4.685357e6,4.685357e6,4.685357e6
"""null_count""",678678.0,248636.0,458014.0,678678.0,284952.0,496592.0,718495.0,214399.0,214399.0,214399.0
"""mean""",-0.495259,-0.114302,-0.342166,-0.466964,-0.01239,-0.005024,0.014165,15.180693,17.797005,21.120708
"""std""",13.58306,7.737659,12.996803,15.182754,3.37395,3.820076,3.993541,6.396895,7.195327,8.502654
"""min""",-203.0,-212.4,-259.4,-224.0,-128.33558,-131.223859,-130.842956,0.0,0.0,0.0
"""25%""",-7.0,-3.0,-6.0,-7.9,-0.996767,-1.161125,-1.245678,12.0,15.0,17.0
"""50%""",0.0,0.0,0.0,-0.1,-0.003548,0.007644,0.027592,15.0,18.0,21.0
"""75%""",6.0,2.9,5.1,7.0,0.974837,1.184119,1.314091,18.9,22.0,26.0
"""max""",203.0,232.9,259.4,259.4,105.572836,106.46976,107.687789,100.0,100.0,100.0


In [20]:
describe_df(df, 100, 110)

statistic,rr_sd,delta_rr10_lag1,delta_rr10_lag6,delta_rr10_lag12,delta_rr50_lag1,delta_rr50_lag6,delta_rr50_lag12,delta_rr90_lag1,delta_rr90_lag6,delta_rr90_lag12
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.636135e6,4.564616e6,4.350629e6,4.128962e6,4.564616e6,4.350629e6,4.128962e6,4.564616e6,4.350629e6,4.128962e6
"""null_count""",263621.0,335140.0,549127.0,770794.0,335140.0,549127.0,770794.0,335140.0,549127.0,770794.0
"""mean""",2.762255,0.009404,0.022417,0.034491,0.002143,0.035041,0.070953,-0.009951,0.044705,0.109257
"""std""",2.062952,2.634907,3.881038,4.444909,2.590159,4.117604,4.81619,3.619172,5.325896,6.080435
"""min""",0.0,-81.0,-80.0,-81.0,-80.0,-82.5,-82.5,-91.5,-100.0,-96.6
"""25%""",1.416072,-1.0,-1.0,-1.9,-1.0,-1.5,-2.0,-1.0,-2.0,-2.1
"""50%""",2.467873,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",3.664587,1.0,1.0,1.9,1.0,1.5,2.0,1.0,2.0,2.9
"""max""",51.618795,80.0,80.1,88.4,80.0,80.0,86.0,90.5,99.4,99.4


In [21]:
describe_df(df, 110, 120)

statistic,delta_rr_sd_lag1,delta_rr_sd_lag6,delta_rr_sd_lag12,sbp10,sbp50,sbp90,sbp_sd,delta_sbp10_lag1,delta_sbp10_lag6,delta_sbp10_lag12
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.528658e6,4.31285e6,4.090695e6,4.51515e6,4.51515e6,4.51515e6,3.564536e6,3.401254e6,3.262747e6,3.087116e6
"""null_count""",371098.0,586906.0,809061.0,384606.0,384606.0,384606.0,1.33522e6,1.498502e6,1.637009e6,1.81264e6
"""mean""",-0.006999,0.007491,0.027899,115.442034,122.583884,130.281263,9.077244,-0.067508,0.201889,0.47857
"""std""",1.590117,1.917535,2.043674,25.608473,25.617299,27.340626,7.619924,14.131445,21.016345,23.338936
"""min""",-50.456135,-50.710908,-49.359164,0.0,0.0,0.0,0.0,-285.3,-284.1,-230.6
"""25%""",-0.644723,-0.801112,-0.865373,98.0,105.0,112.0,4.441586,-5.8,-10.9,-12.3
"""50%""",0.0,0.0,0.0,114.0,121.0,128.2,7.071068,0.0,0.2,0.7
"""75%""",0.639655,0.836636,0.941622,132.0,139.0,147.0,11.294917,5.9,11.2,13.8
"""max""",50.565964,50.565964,50.565964,300.0,300.0,300.0,207.889394,274.0,246.0,244.4


In [22]:
describe_df(df, 120, 130)

statistic,delta_sbp50_lag1,delta_sbp50_lag6,delta_sbp50_lag12,delta_sbp90_lag1,delta_sbp90_lag6,delta_sbp90_lag12,delta_sbp_sd_lag1,delta_sbp_sd_lag6,delta_sbp_sd_lag12,mbp10
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",3.401254e6,3.262747e6,3.087116e6,3.401254e6,3.262747e6,3.087116e6,3.20425e6,3.018295e6,2.835083e6,4.514439e6
"""null_count""",1.498502e6,1.637009e6,1.81264e6,1.498502e6,1.637009e6,1.81264e6,1.695506e6,1.881461e6,2.064673e6,385317.0
"""mean""",-0.125179,0.052656,0.336639,-0.184795,-0.13164,0.141999,-0.054925,-0.094439,-0.057535,77.98467
"""std""",13.255427,20.595036,23.060634,15.360172,22.55229,24.989286,7.193381,8.060263,8.3309,17.362632
"""min""",-278.0,-262.0,-248.5,-275.6,-253.2,-263.4,-203.233945,-198.191185,-203.608116,0.0
"""25%""",-5.0,-10.5,-12.0,-6.0,-11.1,-13.0,-2.712104,-3.12511,-3.313569,66.0
"""50%""",0.0,0.5,0.5,0.0,0.2,0.4,-0.032509,-0.012324,0.027893,77.0
"""75%""",5.5,11.0,13.0,6.0,11.9,14.0,2.620485,3.061073,3.324042,88.7
"""max""",267.0,235.0,253.0,267.0,254.5,226.4,200.038677,203.826888,185.42525,300.0


In [23]:
describe_df(df, 130, 140)

statistic,mbp50,mbp90,mbp_sd,delta_mbp10_lag1,delta_mbp10_lag6,delta_mbp10_lag12,delta_mbp50_lag1,delta_mbp50_lag6,delta_mbp50_lag12,delta_mbp90_lag1
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.514439e6,4.514439e6,3.565571e6,3.401334e6,3.262765e6,3.087131e6,3.401334e6,3.262765e6,3.087131e6,3.401334e6
"""null_count""",385317.0,385317.0,1.334185e6,1.498422e6,1.636991e6,1.812625e6,1.498422e6,1.636991e6,1.812625e6,1.498422e6
"""mean""",82.302116,87.924327,6.828234,-0.05054,0.093933,0.234427,-0.087008,-0.000012,0.142925,-0.127451
"""std""",17.478416,19.87773,7.520599,9.01653,13.250714,14.766061,9.492047,13.970012,15.540202,13.540942
"""min""",0.0,0.0,0.0,-275.6,-274.4,-244.333333,-290.0,-268.0,-244.333333,-292.7
"""25%""",71.0,75.0,2.910918,-3.1,-6.7,-7.9,-3.0,-7.0,-8.0,-4.0
"""50%""",81.0,86.0,4.706338,0.0,0.0,0.1,0.0,0.0,0.0,0.0
"""75%""",93.0,98.2,7.815872,3.1,7.0,8.1,3.5,7.0,8.5,4.0
"""max""",300.0,300.0,200.818326,275.0,241.6,249.9,275.0,240.0,256.5,275.0


In [24]:
describe_df(df, 140, 150)

statistic,delta_mbp90_lag6,delta_mbp90_lag12,delta_mbp_sd_lag1,delta_mbp_sd_lag6,delta_mbp_sd_lag12,dbp10,dbp50,dbp90,dbp_sd,delta_dbp10_lag1
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",3.262765e6,3.087131e6,3.204483e6,3.018518e6,2.835339e6,4.514509e6,4.514509e6,4.514509e6,3.564411e6,3.401078e6
"""null_count""",1.636991e6,1.812625e6,1.695273e6,1.881238e6,2.064417e6,385247.0,385247.0,385247.0,1.335345e6,1.498678e6
"""mean""",-0.088165,0.071155,-0.040199,-0.045659,-0.011629,59.151547,62.611855,67.546517,5.953205,-0.036581
"""std""",18.056392,19.578339,7.374616,8.065464,8.267932,15.779683,16.171093,19.256202,7.890585,7.844691
"""min""",-256.0,-255.1,-184.910742,-190.097909,-194.854186,0.0,0.0,0.0,0.0,-275.0
"""25%""",-7.9,-9.0,-1.983399,-2.230733,-2.363725,48.1,52.0,56.0,2.23354,-2.4
"""50%""",0.0,0.1,-0.02447,-0.002664,0.023314,57.8,61.0,65.0,3.68732,0.0
"""75%""",8.0,9.2,1.915209,2.199808,2.391705,68.0,72.0,76.1,6.363961,2.6
"""max""",245.0,245.6,195.979864,195.373715,188.387377,300.0,300.0,300.0,205.768073,271.4


In [25]:
describe_df(df, 150, 160)

statistic,delta_dbp10_lag6,delta_dbp10_lag12,delta_dbp50_lag1,delta_dbp50_lag6,delta_dbp50_lag12,delta_dbp90_lag1,delta_dbp90_lag6,delta_dbp90_lag12,delta_dbp_sd_lag1,delta_dbp_sd_lag6
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",3.262526e6,3.086904e6,3.401078e6,3.262526e6,3.086904e6,3.401078e6,3.262526e6,3.086904e6,3.204092e6,3.018121e6
"""null_count""",1.63723e6,1.812852e6,1.498678e6,1.63723e6,1.812852e6,1.498678e6,1.63723e6,1.812852e6,1.695664e6,1.881635e6
"""mean""",0.031435,0.089989,-0.065705,-0.026267,0.041006,-0.100148,-0.065552,0.038055,-0.029601,0.002265
"""std""",10.969325,12.105592,8.995,12.419326,13.589667,14.05868,18.006815,19.209163,7.437926,8.267154
"""min""",-292.2,-269.0,-278.0,-289.0,-263.0,-285.0,-263.0,-258.9,-196.316442,-202.424135
"""25%""",-5.0,-6.0,-2.5,-5.0,-6.0,-3.1,-6.1,-7.2,-1.610776,-1.825466
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.01679,0.0
"""75%""",5.0,6.0,2.5,5.5,6.5,3.1,6.1,7.6,1.563583,1.824768
"""max""",261.0,258.4,271.0,258.0,259.0,277.2,260.0,260.0,176.486067,195.946946


In [26]:
describe_df(df, 160, 170)

statistic,delta_dbp_sd_lag12,spo2_10,spo2_50,spo2_90,spo2_sd,delta_spo2_10_lag1,delta_spo2_10_lag6,delta_spo2_10_lag12,delta_spo2_50_lag1,delta_spo2_50_lag6
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",2.834906e6,4.72485e6,4.72485e6,4.72485e6,4.654919e6,4.562282e6,4.349068e6,4.127708e6,4.562282e6,4.349068e6
"""null_count""",2.06485e6,174906.0,174906.0,174906.0,244837.0,337474.0,550688.0,772048.0,337474.0,550688.0
"""mean""",0.05278,96.067343,97.338053,98.181678,1.106929,-0.001523,-0.052597,-0.100307,-0.010029,-0.054872
"""std""",8.488185,4.152595,3.105221,2.567327,1.451012,2.900004,3.768255,4.03818,1.809912,2.524533
"""min""",-203.386707,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,-100.0,-100.0
"""25%""",-1.93851,95.0,96.0,97.0,0.482461,-1.0,-1.0,-1.1,0.0,-1.0
"""50%""",0.027509,97.0,98.0,99.0,0.769474,0.0,0.0,0.0,0.0,0.0
"""75%""",1.995632,98.9,99.0,100.0,1.276042,1.0,1.0,1.0,0.0,1.0
"""max""",191.570143,100.0,100.0,100.0,70.710678,100.0,100.0,100.0,100.0,100.0


In [27]:
describe_df(df, 170, 180)

statistic,delta_spo2_50_lag12,delta_spo2_90_lag1,delta_spo2_90_lag6,delta_spo2_90_lag12,delta_spo2_sd_lag1,delta_spo2_sd_lag6,delta_spo2_sd_lag12,bp_invasive,wbc,hemoglobin
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.127708e6,4.562282e6,4.349068e6,4.127708e6,4.506737e6,4.289045e6,4.066808e6,4.81265e6,4.00208e6,3.990096e6
"""null_count""",772048.0,337474.0,550688.0,772048.0,393019.0,610711.0,832948.0,87106.0,897676.0,909660.0
"""mean""",-0.090109,-0.011025,-0.046626,-0.0712,-0.005316,0.001768,0.012067,0.671283,10.818384,10.254978
"""std""",2.763758,1.459746,2.039126,2.247491,1.341237,1.549888,1.612181,0.469747,7.189804,2.096217
"""min""",-100.0,-100.0,-100.0,-100.0,-69.710678,-63.920195,-62.959967,0.0,0.0,-0.48
"""25%""",-1.0,0.0,-1.0,-1.0,-0.279742,-0.354152,-0.377214,0.0,6.91,8.7
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,9.56,10.0
"""75%""",1.0,0.0,1.0,1.0,0.274624,0.374966,0.421741,1.0,13.1,11.5
"""max""",100.0,100.0,100.0,100.0,68.882931,69.915767,69.865118,1.0,636.5,22.6


In [28]:
describe_df(df, 180, 190)

statistic,platelet,sodium,creatinine,potassium,aspartate_aminotransferase,blood_urea_nitrogen,alanine_aminotransferase,total_bilirubin,crp,albumin
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.000982e6,3.999708e6,3.99035e6,3.997175e6,3.939408e6,3.981924e6,3.939553e6,3.910161e6,3.762644e6,3.685242e6
"""null_count""",898774.0,900048.0,909406.0,902581.0,960348.0,917832.0,960203.0,989595.0,1.137112e6,1.214514e6
"""mean""",193.924194,140.173073,1.437072,4.005867,108.138481,29.920707,75.324478,1.448995,8.47933,2.501873
"""std""",126.114939,5.950658,1.632597,0.62151,532.309373,23.250015,257.883091,3.111458,8.379983,0.684232
"""min""",0.0,48.0,0.01,0.4,2.0,1.0,0.0,0.0,0.0,0.1
"""25%""",105.0,137.0,0.6,3.6,22.0,14.0,15.0,0.45,1.969,2.0
"""50%""",173.0,140.0,0.84,4.0,34.0,22.8,27.0,0.7,5.858,2.4
"""75%""",254.0,143.0,1.5,4.3,65.0,38.1,56.0,1.2,12.61,2.9
"""max""",1927.0,198.0,37.97,29.4,50600.0,439.4,11202.0,134.7,76.6,6.2


In [29]:
describe_df(df, 190, 200)

statistic,rass_score,respiration_24hours,coagulation_24hours,liver_24hours,cardiovascular_24hours,cns_24hours,renal_24hours,sofa_24hours,input_amount,cumsum_in
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.397116e6,4.488377e6,4.02413e6,3.938184e6,4.870831e6,4.725174e6,4.179444e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""",502640.0,411379.0,875626.0,961572.0,28925.0,174582.0,720312.0,0.0,0.0,0.0
"""mean""",-1.758012,1.578014,0.762408,0.448751,1.312955,1.599345,1.159061,6.2686,106.622843,29637.099879
"""std""",1.982199,1.087374,1.072359,0.875625,1.247609,1.468094,1.578254,4.099557,278.237505,203432.990625
"""min""",-5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",-4.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,20.0,3521.333333
"""50%""",-1.0,2.0,0.0,0.0,1.0,1.0,0.0,6.0,60.0,10059.633333
"""75%""",0.0,2.0,1.0,1.0,2.0,3.0,3.0,9.0,112.3,25312.09
"""max""",4.0,4.0,4.0,4.0,4.0,4.0,4.0,24.0,100032.0,8.5919e6


In [30]:
describe_df(df, 200, 210)

statistic,window_total_output,cumsum_out,cumsum_balance,active_ingredient_propofol,active_ingredient_noradrenaline,active_ingredient_fentanyl,active_ingredient_dexmedetomidine,active_ingredient_dopamine,active_ingredient_midazolam,active_ingredient_dobutamine
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",69.970926,15894.007728,13743.092151,15.448822,0.061543,0.009946,0.015687,0.561606,0.555039,0.749414
"""std""",95.259701,28489.143056,188600.979066,45.658906,0.206849,0.025414,0.094112,3.394437,2.351753,3.394129
"""min""",0.0,0.0,-428443.540333,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",24.48,1957.917,-206.666,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",50.583,6690.005,2029.086,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",91.25,18560.869,7795.135273,0.0,0.0,0.015,0.0,0.0,0.0,0.0
"""max""",28280.64,541423.116,8.0505e6,4000.0,13.24,11.0,5.0,383.5,80.958333,163.183929


In [31]:
describe_df(df, 210, 222)

statistic,active_ingredient_landiolol,active_ingredient_vasopressin,active_ingredient_remifentanil,active_ingredient_thiamylal,active_ingredient_adrenaline,active_ingredient_vecuronium,active_ingredient_rocuronium,transfusion_rbc,transfusion_platelet,transfusion_plasma,transfusion_albumin
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6,4.899756e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.226204,0.032604,0.001091,0.172664,0.002298,0.01115,0.321134,2.106507,0.386908,1.330155,0.980675
"""std""",1.729248,0.257309,0.011609,7.019047,0.063077,0.175544,3.518124,23.475368,8.570139,23.409596,12.25906
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""",75.0,31.372549,0.75,2500.0,32.2,53.0,200.0,7361.7506,2828.571384,5091.428585,4250.0


## Cohort Splilt

In [32]:
# 全施設を8:2にsplitする方針
df_development = df.filter(pl.col('hospital_id').is_in([1, 2, 3, 5, 6, 7, 8, 9, 10, 11]))
del df # メモリ節約

In [33]:
# record数と症例数を確認
print('Number of records (model development cohort):', len(df_development))
print('Number of icu_stay_id (model development cohort):', len(df_development['icu_stay_id'].unique()))
print('Average number of records per icu_stay_id (model development cohort):', len(df_development) / len(df_development['icu_stay_id'].unique()))

Number of records (model development cohort): 4899756
Number of icu_stay_id (model development cohort): 41300
Average number of records per icu_stay_id (model development cohort): 118.6381598062954


In [34]:
df_development.write_csv(os.path.join(data_dir, df_development_file_name))